# Dedekind DSL: Beginner Tier

This notebook sketches a **beginner-friendly DSL** for set operations using SQL/Python-like syntax.

**Design philosophy**: Use familiar language constructs (`where`, `select`, method chaining) to make set logic accessible to users unfamiliar with mathematical notation.

## Scope & Prototype Status

This is a **prototype DSL shim** demonstrating desired ergonomics, not the final API. The goal is to show:
- **Intensional** (symbolic) definitions: define sets by rules, not values
- **Extensional** (realized) computation: evaluate rules to get concrete results

Outputs may be approximate; correctness refinement is a follow-up item (issue #241).

In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing pyproject.toml")

repo_root = find_repo_root(Path.cwd())

try:
    import dedekind
except ModuleNotFoundError as err:
    raise ModuleNotFoundError(
        "dedekind is not installed in this notebook kernel. "
        f"From {repo_root}, run `python -m pip install -e .` "
        "or use `make integration-test` for the CI-style execution path."
    ) from err

print("beginner-tier: dedekind import ok")

In [ ]:
class SetDef:
    """Beginner-tier set definition: intensional (symbolic) representation.
    
    This class wraps a predicate and a universe for lazy evaluation.
    """

    def __init__(self, values=None, predicate=None, universe=None):
        # If values are given, store them directly (eager)
        self._values = list(values) if values is not None else None
        # Otherwise store a symbolic definition
        self._predicate = predicate
        self._universe = universe or set()

    @staticmethod
    def from_list(items):
        """Create an extensional (concrete) set from a list."""
        return SetDef(values=items)

    @staticmethod
    def from_range(start, stop=None, step=1):
        """Create a set from a numeric range."""
        if stop is None:
            stop = start
            start = 0
        return SetDef(values=range(start, stop, step))

    def where(self, condition):
        """Filter: keep elements satisfying the condition.
        
        This returns an intensional (unevaluated) set definition.
        """
        if self._values is not None:
            # If we have concrete values, filter them
            filtered = [v for v in self._values if condition(v)]
            return SetDef(values=filtered)
        else:
            # Otherwise return a symbolic definition
            return SetDef(
                predicate=lambda x: (self._predicate(x) if self._predicate else True) and condition(x),
                universe=self._universe
            )

    def select(self, mapping):
        """Transform: apply a function to each element (map/image).
        
        This is the set-builder equivalent of 'select'.
        """
        if self._values is not None:
            mapped = [mapping(v) for v in self._values]
            return SetDef(values=mapped)
        else:
            # Cannot easily map a symbolic set; fall back to identity for demo
            return self

    def union(self, other):
        """Set union: A ∪ B."""
        if self._values is not None and other._values is not None:
            result = list(set(self._values) | set(other._values))
            return SetDef(values=result)
        return self  # Fallback for symbolic

    def intersect(self, other):
        """Set intersection: A ∩ B."""
        if self._values is not None and other._values is not None:
            result = list(set(self._values) & set(other._values))
            return SetDef(values=result)
        return self  # Fallback for symbolic

    def minus(self, other):
        """Set difference: A \ B."""
        if self._values is not None and other._values is not None:
            result = list(set(self._values) - set(other._values))
            return SetDef(values=result)
        return self  # Fallback for symbolic

    def realize(self, *, ordered=True):
        """Extensional (concrete) realization: evaluate to get actual values.
        
        This transitions from intensional (symbolic) to extensional (computed).
        """
        if self._values is not None:
            if ordered:
                return dedekind.ordered_set_roundtrip(self._values)
            else:
                return dedekind.unordered_set_roundtrip(self._values)
        # Symbolic sets cannot be realized yet
        raise NotImplementedError("Symbolic set realization not yet implemented")

    def __repr__(self):
        if self._values is not None:
            return f"SetDef({self._values})"
        return f"SetDef(symbolic, predicate={self._predicate})"


print("beginner-tier: shim types ready")

## Part 1: Intensional (Symbolic) Definitions

Define sets using rules and conditions. These are not yet evaluated—they exist as definitions.

In [ ]:
# Define sets symbolically (not computed yet)

# Set A: the integers from 1 to 5
A = SetDef.from_list([1, 2, 3, 4, 5])
print(f"A (symbolic): {A}")

# Set B: the integers from 3 to 7
B = SetDef.from_list([3, 4, 5, 6, 7])
print(f"B (symbolic): {B}")

# Set E: numbers from 1 to 20 where x is even
E = SetDef.from_range(1, 21).where(lambda x: x % 2 == 0)
print(f"E = {{x ∈ [1..20] | x is even}} (symbolic): {E}")

# Set S: squares of even numbers from 1 to 10
S = SetDef.from_range(1, 11).where(lambda x: x % 2 == 0).select(lambda x: x * x)
print(f"S = {{x² | x ∈ [1..10], x is even}} (symbolic): {S}")

## Part 2: Extensional (Realized) Computation

Now evaluate these definitions to get concrete results. Call `.realize()` to transition from intensional to extensional.

In [ ]:
# Realize (compute) the sets defined above

# Union and intersection
union_AB = A.union(B).realize()
intersect_AB = A.intersect(B).realize()
diff_AB = A.minus(B).realize()

print("=== Extensional Results ===")
print(f"A ∪ B = {union_AB}")
print(f"A ∩ B = {intersect_AB}")
print(f"A \\ B = {diff_AB}")

# Filtered set
evens = E.realize()
print(f"{{x ∈ [1..20] | x is even}} = {evens}")

# Mapped set
squares = S.realize()
print(f"{{x² | x ∈ [1..10], x is even}} = {squares}")

## Validation

Verify that the realized sets match expectations.

In [ ]:
# Assertions on computed results
assert union_AB == [1, 2, 3, 4, 5, 6, 7], f"Union failed: {union_AB}"
assert intersect_AB == [3, 4, 5], f"Intersection failed: {intersect_AB}"
assert diff_AB == [1, 2], f"Difference failed: {diff_AB}"
assert evens == [2, 4, 6, 8, 10, 12, 14, 16, 18, 20], f"Evens failed: {evens}"
assert squares == [4, 16, 36, 64, 100], f"Squares failed: {squares}"

print("✓ All assertions passed")
print("notebook-03-ok")